# ⚛️ Quantum VQC Classifier on Simple Dataset

This notebook trains a Variational Quantum Classifier (VQC) on the same `make_moons` dataset used in the classical baseline.  
It uses a custom noise model.

In [ ]:
# Imports

import numpy as np
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score

from qiskit.circuit.library import ZZFeatureMap, RealAmplitudes
from qiskit_algorithms.optimizers import COBYLA
from qiskit_machine_learning.algorithms import VQC

from qiskit_aer.noise import NoiseModel, depolarizing_error

from qiskit_aer import AerSimulator
from qiskit.primitives import Sampler

In [ ]:
# Seed and Data

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

X, y = make_moons(n_samples=200, noise=0.1, random_state=RANDOM_SEED)
scaler = MinMaxScaler(feature_range=(0, np.pi))
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.3, random_state=RANDOM_SEED
)

In [ ]:
# Custom Noise Model version 1
#noise_model = NoiseModel()
#noise_model.add_all_qubit_quantum_error(depolarizing_error(0.004, 1), ['rx', 'ry', 'rz', 'h', 'x', 'y', 'z'])
#noise_model.add_all_qubit_quantum_error(depolarizing_error(0.025, 2), ['cx', 'cz', 'swap'])

#simulator = AerSimulator(noise_model=noise_model, method='density_matrix')
#sampler = Sampler(options={'backend': simulator})

#print("✅ Custom Noise Model Initialized (1Q: 0.4%, 2Q: 2.5%)")


from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, ReadoutError

# Custom Noise Model version 2
# ✅ This noise model includes:
# - Uniform depolarizing noise on all 1Q and 2Q gates
# - Readout error on all qubits
# - Qubit-specific gate error rates (optional)
# - Optional coupling map to simulate limited connectivity

# 🚫 This model does NOT include:
# - Crosstalk between qubits
# - Time-dependent decoherence
# - Calibration-derived noise from real hardware
# - Topology-aware gate durations or delays

# --- Enhanced Custom Noise Model (Qiskit V1-Compatible) ---

# Create a noise model
noise_model = NoiseModel()

# Add uniform depolarizing noise to all 1-qubit gates
noise_model.add_all_qubit_quantum_error(depolarizing_error(0.004, 1), ['rx', 'ry', 'rz', 'h', 'x', 'y', 'z'])

# Add uniform depolarizing noise to all 2-qubit gates
noise_model.add_all_qubit_quantum_error(depolarizing_error(0.025, 2), ['cx', 'cz', 'swap'])

# Add readout error to all qubits
readout_error = ReadoutError([[0.9, 0.1], [0.2, 0.8]])  # 10% flip on 0, 20% flip on 1
noise_model.add_all_qubit_readout_error(readout_error)

# Optional: Qubit-specific gate errors (currently commented out)
# These lines simulate non-uniform noise across qubits, reflecting realistic calibration drift or hardware variability.
# For example, Qubit 0 has lower error (0.5%) than Qubit 1 (1.0%) on 'x' gates.
# This helps illustrate how individual qubit fidelity can impact circuit performance.
# Uncomment to explore asymmetric noise effects or simulate "bad qubit" scenarios.

# noise_model.add_quantum_error(depolarizing_error(0.005, 1), ['x'], [0])  # Qubit 0
# noise_model.add_quantum_error(depolarizing_error(0.01, 1), ['x'], [1])  # Qubit 1

# Initialize simulator with noise model
simulator = AerSimulator(noise_model=noise_model, method='density_matrix')
sampler = Sampler(options={'backend': simulator})

print("✅ Enhanced Custom Noise Model Loaded")

In [ ]:
# Quantum Circuit Setup

feature_map = ZZFeatureMap(feature_dimension=X_train.shape[1], reps=2, entanglement='linear')
ansatz = RealAmplitudes(feature_map.num_qubits, reps=3)
optimizer = COBYLA(maxiter=100)

In [ ]:
# VQC Initialization and Training
vqc = VQC(
    sampler=sampler,
    ansatz=ansatz,
    feature_map=feature_map,
    optimizer=optimizer,
    callback=None,
)

# 🔍 Print the quantum circuit structure
print("🔍 Quantum Circuit Structure (VQC):")
print(vqc.circuit.draw(output='text'))
vqc.circuit.draw(output='mpl')

# 🧠 Inspect individual gates for advanced learners
for i, gate in enumerate(vqc.circuit.data):
    print(f"{i}: {gate}")

# 🧩 Decomposed circuit diagram for gate-level clarity
vqc.circuit.decompose().draw(output='mpl')

# 📏 Analyze circuit complexity
decomposed = vqc.circuit.decompose()
print("Circuit depth:", decomposed.depth())
print("Gate counts:", decomposed.count_ops())

print("🚀 Starting VQC Training...")
vqc.fit(X_train, y_train)
print("🚀 Starting VQC Training DONE...")

### 🔍 Gate-Level Inspection

Each line printed above shows a gate applied in the circuit, including:
- The **gate type** (e.g., `ry`, `cx`, `rz`)
- The **target qubits**
- The **execution order**

This helps for debugging and optimization.
---

### 🧩 Decomposed Circuit Diagram

The final diagram reveals the **low-level gates** that make up composite instructions like `RY` and `CX`.

> 🧠 Why this matters: Decomposition shows the true depth and complexity of the circuit, which affects noise sensitivity and execution time on the simulator.

### 📏 Circuit Complexity Metrics

To understand how circuit structure affects performance, we measure:

- **Depth**: The number of sequential gate layers. Deeper circuits are more sensitive to noise and take longer to simulate.
- **Gate counts**: The total number of each gate type (e.g., `cx`, `ry`, `rz`). More gates = more opportunities for error.

> 🧠 Why this matters: Even on simulators, deeper and more complex circuits take longer to run and are more affected by noise models. These metrics help us reason about scalability and robustness.

In [ ]:
#Evaluation
y_pred_vqc = vqc.predict(X_test)
accuracy_vqc = accuracy_score(y_test, y_pred_vqc)

print("-" * 50)
print(f"✅ Quantum VQC Accuracy (Reps={ansatz.reps}): {accuracy_vqc*100:.2f}% (WITH CUSTOM NOISE)")
print("-" * 50)

# 🔍 Reflection
- How does the quantum model's accuracy compare to the classical baseline?
- What role does noise play in this setup?
- Is the model capacity (ansatz depth) a limiting factor?